In [190]:
import random
from typing import Dict, List
import nltk
from nltk.corpus import wordnet as wn
from wordfreq import top_n_list
nltk.download("wordnet")
import spacy
from collections import defaultdict
from tqdm import tqdm
import inflect

p = inflect.engine()
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mgrom\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [193]:
VOWELS = "aeiou"
CONSONANTS = "bcdfghjklmnpqrstvwxyz"

def set_seed(seed: int):
    random.seed(seed)

def generate_wordlike_token(min_len=3, max_len=8):
    length = random.randint(min_len, max_len)
    token = []
    use_consonant = random.choice([True, False])
    for _ in range(length):
        if use_consonant:
            token.append(random.choice(CONSONANTS))
        else:
            token.append(random.choice(VOWELS))
        if random.random() < 0.8:
            use_consonant = not use_consonant
    return "".join(token)

def build_full_vocab(max_words=5000, min_len=3, max_len=8):
    freq_words = top_n_list("en", max_words)
    freq_words = [w.lower() for w in freq_words]

    pos_map = defaultdict(set)

    for word in tqdm(freq_words, desc="Processing words"):
        doc = nlp(word)
        for token in doc:
            if token.pos_ == "NOUN":
                pos_map["nouns"].add(word)
            elif token.pos_ == "VERB":
                pos_map["verbs"].add(word)
            elif token.pos_ == "ADJ":
                pos_map["adjectives"].add(word)

    # length filter
    for k in pos_map:
        pos_map[k] = {
            w for w in pos_map[k]
            if min_len <= len(w) <= max_len
        }

    return {k: sorted(list(v)) for k, v in pos_map.items()}

def sample_vocab(full_vocab, n_nouns=30, n_verbs=15, n_adjectives=10):
    return {
        "nouns": random.sample(full_vocab["nouns"], n_nouns),
        "verbs": random.sample(full_vocab["verbs"], n_verbs),
        "adjectives": random.sample(full_vocab["adjectives"], n_adjectives),
    }

def create_mapping(vocab: Dict[str, List[str]]):
    mapping = {}
    used_tokens = set()
    words = vocab["nouns"] + vocab["verbs"] + vocab.get("adjectives", [])
    for word in words:
        token = generate_wordlike_token()
        while token in used_tokens:
            token = generate_wordlike_token()
        mapping[word] = token
        used_tokens.add(token)
    return mapping

def maybe_plural(word):
    if random.random() < 0.5:
        return p.plural(word)
    return word

def maybe_negate(verb):
    if random.random() < 0.5:
        return f"not {verb}"
    return verb

def generate_sentence_lvl1(vocab):
    subj = random.choice(vocab["nouns"])
    obj = random.choice([n for n in vocab["nouns"] if n != subj])
    verb = random.choice(vocab["verbs"])
    tokens = [subj, verb, obj]
    return tokens, " ".join(tokens)

def generate_sentence_lvl2(vocab):
    nouns = vocab["nouns"]
    verbs = vocab["verbs"]
    adjectives = vocab.get("adjectives", [])

    subj_base = random.choice(nouns)
    obj_base = random.choice([n for n in nouns if n != subj_base])
    verb = random.choice(verbs)

    tokens = []

    # ----- Subject -----
    adj1 = random.choice(adjectives)
    tokens.append(adj1)

    subj_base = maybe_plural(subj_base)
    tokens.append(subj_base)

    # ----- Verb -----
    tokens.append(verb)

    # ----- Object -----
    adj2 = random.choice(adjectives)
    tokens.append(adj2)

    obj_base = maybe_plural(obj_base)
    tokens.append(obj_base)

    return tokens, " ".join(tokens)


def generate_sentence_lvl3(vocab):
    nouns = vocab["nouns"]
    verbs = vocab["verbs"]
    adjectives = vocab.get("adjectives", [])

    subj_base = random.choice(nouns)
    obj_base = random.choice([n for n in nouns if n != subj_base])
    verb = random.choice(verbs)

    tokens = []

    # ----- Subject -----
    adj1 = random.choice(adjectives)
    tokens.append(adj1)

    subj_base = maybe_plural(subj_base)
    tokens.append(subj_base)

    # ----- Negation + Verb -----
    verb = maybe_negate(verb)
    tokens.append(verb)

    # ----- Object -----
    adj2 = random.choice(adjectives)
    tokens.append(adj2)

    obj_base = maybe_plural(obj_base)
    tokens.append(obj_base)

    return tokens, " ".join(tokens)

def generate_sentence_lvl4(vocab):
    nouns = vocab["nouns"]
    verbs = vocab["verbs"]
    adjectives = vocab.get("adjectives", [])

    subj_base = random.choice(nouns)
    obj_base = random.choice([n for n in nouns if n != subj_base])
    verb = random.choice(verbs)

    tokens = []

    # ----- Subject phrase -----
    subj_adjs = []
    if adjectives:
        k = random.randint(1, min(2, len(adjectives)))
        subj_adjs = random.sample(adjectives, k=k)

    subj_surface = " ".join(subj_adjs + [subj_base])
    tokens.extend(subj_adjs)

    if random.random() < 0.5:
        subj_surface += "-pl"
    tokens.append(subj_base)

    # ----- Negation + Verb -----
    if random.random() < 0.5:
        sentence_verb = f"not {verb}"
        tokens.append("NOT")
    else:
        sentence_verb = verb

    tokens.append(verb)

    # ----- Object phrase -----
    obj_adjs = []
    if adjectives:
        k = random.randint(1, min(2, len(adjectives)))
        obj_adjs = random.sample(adjectives, k=k)

    obj_surface = " ".join(obj_adjs + [obj_base])
    tokens.extend(obj_adjs)

    if random.random() < 0.5:
        obj_surface += "-pl"
    tokens.append(obj_base)

    # ----- Sentence -----
    sentence = f"{subj_surface} {sentence_verb} {obj_surface}"

    return sentence, tokens

def generate_sentence_lvl5(vocab):
    nouns = vocab["nouns"]
    verbs = vocab["verbs"]
    adjectives = vocab.get("adjectives", [])

    def build_noun_phrase():
        base = random.choice(nouns)
        adjs = []
        if adjectives:
            k = random.randint(0, min(2, len(adjectives)))
            adjs = random.sample(adjectives, k=k)

        phrase = " ".join(adjs + [base])
        tokens_local = adjs.copy()

        if random.random() < 0.5:
            phrase += "-pl"
        tokens_local.append(base)

        return phrase, tokens_local

    def build_verb_phrase():
        verb = random.choice(verbs)
        tokens_local = []

        if random.random() < 0.5:
            phrase = f"not {verb}"
            tokens_local.append("NOT")
        else:
            phrase = verb

        tokens_local.append(verb)
        return phrase, tokens_local

    tokens = []

    # First clause
    subj1, t1 = build_noun_phrase()
    verb1, t2 = build_verb_phrase()
    obj1, t3 = build_noun_phrase()

    tokens.extend(t1)
    tokens.extend(t2)
    tokens.extend(t3)

    sentence = f"{subj1} {verb1} {obj1}"

    # Maybe add second clause
    if random.random() < 0.5:
        subj2, t4 = build_noun_phrase()
        verb2, t5 = build_verb_phrase()
        obj2, t6 = build_noun_phrase()

        connector = random.choice(["and", "then", "that"])

        sentence = f"{sentence} {connector} {subj2} {verb2} {obj2}"

        tokens.extend(t4)
        tokens.extend(t5)
        tokens.extend(t6)

    return sentence, tokens

In [194]:
full_vocab = build_full_vocab(max_words=1000)

Processing words: 100%|██████████| 1000/1000 [00:05<00:00, 181.83it/s]


In [195]:
sampled_vocab = sample_vocab(full_vocab)

In [196]:
generate_sentence_lvl1(sampled_vocab)

(['people', 'check', 'society'], 'people check society')

In [197]:
generate_sentence_lvl2(sampled_vocab)

(['medical', 'test', 'check', 'entire', 'money'],
 'medical test check entire money')

In [198]:
generate_sentence_lvl3(sampled_vocab)

(['next', 'monies', 'not heard', 'serious', 'sources'],
 'next monies not heard serious sources')

In [226]:
def translate(tokens: List[str], mapping: Dict[str,str]):
    isl_tokens = []
    for t in tokens:
        isl_tokens.append(mapping.get(t))
    return " ".join(isl_tokens)

def generate_task(full_vocab, n_train=5, n_test=1):
    vocab = sample_vocab(full_vocab)
    mapping = create_mapping(vocab)
    seen_sentences = set()
    train, test = [], []

    # Collect all words used in train sentences
    train_tokens_set = set()

    def sample_unique():
        while True:
            tokens, sentence = generate_sentence_lvl1(vocab)
            if sentence not in seen_sentences:
                seen_sentences.add(sentence)
                return tokens, sentence

    # Generate training data
    for _ in range(n_train):
        tokens, sentence = sample_unique()
        isl = translate(tokens, mapping)
        train.append((sentence, isl))
        train_tokens_set.update(tokens)

    # Generate test data
    while len(test) < n_test:
        tokens, sentence = sample_unique()
        # Only accept test sentence if all tokens are in training set
        if all(t in train_tokens_set for t in tokens):
            isl = translate(tokens, mapping)
            test.append((sentence, isl))

    return {
        "vocab": vocab,
        "mapping": mapping,
        "train": train,
        "test": test
    }

In [227]:
task = generate_task(full_vocab, n_train=5, n_test=1)

In [228]:
task["train"]

[('service check week', 'uleaqjo apmiya oci'),
 ('piece included friend', 'fef enipeato kes'),
 ('street thought days', 'xmi wuuy goo'),
 ('service created piece', 'uleaqjo olixoze fef'),
 ('meeting looks example', 'naiipev pvon niph')]

In [229]:
task["test"]

[('service included days', 'uleaqjo enipeato goo')]